In [1]:
# ======================================================================
# GOOGLE COLAB — FULL PIPELINE (single cell)
# ======================================================================
# Mounts Drive, creates the folder, generates data, trains models,
# produces 12 slide visuals + tables.

# ----------------------------------------------------------------------
# STEP 0 — Mount Google Drive & set working directory
# ----------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/HealthCare_/CTR_Recruitment/DataSource"
os.makedirs(BASE_DIR, exist_ok=True)
os.chdir(BASE_DIR)
print(f"📁 Working directory: {os.getcwd()}")

# ----------------------------------------------------------------------
# STEP 1 — Imports
# ----------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, timedelta
from matplotlib.ticker import MaxNLocator
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

np.random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
})
BRAND = ["#1F4E79", "#2E75B6", "#9DC3E6", "#F4B183", "#C00000", "#548235", "#7030A0"]

print("=" * 70)
print("CLINICAL TRIAL RECRUITMENT — FULL PIPELINE")
print("=" * 70)

# ======================================================================
# STAGE 1 — CATALOGS
# ======================================================================
PROTOCOLS = {
    "PROTO-A_ONCO": dict(
        therapeutic_area="Oncology", phase="III", indication="NSCLC",
        sponsor_type="Pharma", protocol_complexity_score=8, num_visits=18,
        num_inclusion_criteria=12, num_exclusion_criteria=15,
        biomarker_required=True, prior_therapy_restriction=True,
        placebo_controlled=True,
    ),
    "PROTO-B_CARDIO": dict(
        therapeutic_area="Cardiology", phase="II", indication="Heart Failure",
        sponsor_type="Biotech", protocol_complexity_score=5, num_visits=9,
        num_inclusion_criteria=8, num_exclusion_criteria=10,
        biomarker_required=False, prior_therapy_restriction=True,
        placebo_controlled=True,
    ),
    "PROTO-C_NEURO": dict(
        therapeutic_area="Neurology", phase="II", indication="Alzheimer's",
        sponsor_type="Academic", protocol_complexity_score=7, num_visits=14,
        num_inclusion_criteria=10, num_exclusion_criteria=12,
        biomarker_required=True, prior_therapy_restriction=False,
        placebo_controlled=False,
    ),
}

AD_CHANNELS = {
    "adv_poster":                 dict(type="passive", cost=200,  reach="low",    effectiveness=0.15),
    "adv_pamphlet":               dict(type="passive", cost=150,  reach="low",    effectiveness=0.12),
    "adv_email":                  dict(type="active",  cost=500,  reach="high",   effectiveness=0.45),
    "adv_online":                 dict(type="active",  cost=1200, reach="high",   effectiveness=0.35),
    "adv_newsletter":             dict(type="passive", cost=300,  reach="medium", effectiveness=0.20),
    "adv_newspaper":              dict(type="passive", cost=800,  reach="medium", effectiveness=0.10),
    "adv_radio":                  dict(type="passive", cost=1500, reach="medium", effectiveness=0.08),
    "adv_tv":                     dict(type="passive", cost=5000, reach="high",   effectiveness=0.12),
    "adv_community_presentation": dict(type="active",  cost=400,  reach="medium", effectiveness=0.40),
    "adv_other":                  dict(type="active",  cost=600,  reach="varies", effectiveness=0.30),
}

CHANNELS = list(AD_CHANNELS.keys())
CHANNEL_LABEL = {
    "adv_poster": "Poster", "adv_pamphlet": "Pamphlet", "adv_email": "Email",
    "adv_online": "Online", "adv_newsletter": "Newsletter", "adv_newspaper": "Newspaper",
    "adv_radio": "Radio", "adv_tv": "TV",
    "adv_community_presentation": "Community", "adv_other": "Other",
}
PROTO_LABEL = {
    "PROTO-A_ONCO":  "A · Oncology (NSCLC, Ph III)",
    "PROTO-B_CARDIO":"B · Cardiology (HF, Ph II)",
    "PROTO-C_NEURO": "C · Neurology (AD, Ph II)",
}

COUNTRY_REG = {
    "USA":       dict(region="North America", authority="FDA",   ec_default="IRB", import_license=False),
    "Germany":   dict(region="Europe",        authority="EMA",   ec_default="CEC", import_license=True),
    "UK":        dict(region="Europe",        authority="MHRA",  ec_default="LEC", import_license=True),
    "Japan":     dict(region="Asia",          authority="PMDA",  ec_default="CEC", import_license=True),
    "Brazil":    dict(region="LATAM",         authority="Other", ec_default="LEC", import_license=True),
    "India":     dict(region="Asia",          authority="Other", ec_default="LEC", import_license=True),
    "Poland":    dict(region="Europe",        authority="EMA",   ec_default="LEC", import_license=True),
    "Spain":     dict(region="Europe",        authority="EMA",   ec_default="LEC", import_license=True),
    "Canada":    dict(region="North America", authority="Other", ec_default="IRB", import_license=False),
    "Australia": dict(region="APAC",          authority="Other", ec_default="LEC", import_license=True),
}
COUNTRIES = list(COUNTRY_REG.keys())
COUNTRY_PROBS = np.array([0.22, 0.14, 0.10, 0.10, 0.08, 0.08, 0.10, 0.08, 0.05, 0.05])
COUNTRY_PROBS = COUNTRY_PROBS / COUNTRY_PROBS.sum()

# ======================================================================
# STAGE 2 — SYNTHETIC DATA GENERATION
# ======================================================================
print("\n[Stage 2] Generating synthetic dataset ...")

N_ROWS = 3000

def ts(d):
    return pd.Timestamp(d)

def sample_ec_timeline(ec_type):
    if ec_type == "LEC":
        return np.random.randint(10, 45), np.random.randint(20, 70)
    elif ec_type == "CEC":
        return np.random.randint(20, 90), np.random.randint(45, 150)
    else:
        return np.random.randint(15, 60), np.random.randint(30, 100)

def sample_advertisement(complexity):
    active_bias = float(np.clip(0.15 + 0.05 * (5 - abs(complexity - 5)), 0.10, 0.45))
    passive_bias = 0.55
    probs = {
        "adv_poster": 0.75, "adv_pamphlet": 0.65,
        "adv_email": active_bias + 0.20, "adv_online": active_bias + 0.10,
        "adv_newsletter": passive_bias * 0.5, "adv_newspaper": passive_bias * 0.35,
        "adv_radio": passive_bias * 0.18, "adv_tv": passive_bias * 0.12,
        "adv_community_presentation": active_bias + 0.05, "adv_other": 0.15,
    }
    return {k: int(np.random.binomial(1, min(v, 0.95))) for k, v in probs.items()}

def generate_row(i):
    pid = np.random.choice(list(PROTOCOLS.keys()))
    p = PROTOCOLS[pid]

    country = np.random.choice(COUNTRIES, p=COUNTRY_PROBS)
    reg = COUNTRY_REG[country]
    region = reg["region"]
    authority = reg["authority"]
    ec_type = reg["ec_default"]
    if np.random.rand() < 0.25:
        ec_type = "LEC" if ec_type == "CEC" else "CEC"
    import_license = reg["import_license"]

    pi_exp = int(np.random.randint(2, 25))
    pi_prior = int(np.random.poisson(3))
    coord = int(np.random.randint(1, 6))
    gcp = int(np.random.binomial(1, 0.92))

    base_pool = {"NSCLC": 420, "Heart Failure": 980, "Alzheimer's": 260}[p["indication"]]
    penalty = 0.35 if p["biomarker_required"] else 1.0
    eligible = int(max(np.random.normal(base_pool * penalty, base_pool * 0.15), 20))
    competing = int(np.random.poisson(4))

    dur = float(round(np.random.uniform(1.0, 4.0), 1))
    est_start = ts(date(2021, 1, 1) + timedelta(days=int(np.random.randint(0, 1000))))

    ec_sub_delay, ec_app_delay = sample_ec_timeline(ec_type)
    ec_sub = est_start - pd.Timedelta(days=ec_sub_delay + ec_app_delay)
    ec_app = ec_sub + pd.Timedelta(days=ec_app_delay)

    reg_sub_delay = np.random.randint(10, 60)
    reg_app_delay = np.random.randint(30, 180)
    reg_sub = ec_sub - pd.Timedelta(days=reg_sub_delay)
    reg_app = reg_sub + pd.Timedelta(days=reg_app_delay)

    import_delay = np.random.randint(30, 90) if import_license else 0
    amendments = int(np.random.poisson(2))

    startup = np.random.randint(5, 30)
    fpi_extra = import_delay + amendments * np.random.randint(3, 10)
    actual_fpi = max(ec_app, reg_app) + pd.Timedelta(
        days=startup + fpi_extra + int(np.random.normal(10, 15))
    )

    planned = int(dur * 365)
    shrink = fpi_extra + ec_app_delay // 3
    window = int(max(planned - shrink, 90))
    actual_lpi = actual_fpi + pd.Timedelta(days=window)
    est_completion = actual_lpi + pd.Timedelta(days=int(np.random.normal(0, 30)))

    fpi_delay = int((actual_fpi - est_start).days)
    slip = int((actual_lpi - est_completion).days)
    quarter = int(actual_fpi.quarter)
    year = int(actual_fpi.year)

    adv = sample_advertisement(p["protocol_complexity_score"])
    adv_other_specify = "Patient registry outreach" if adv["adv_other"] else ""

    active_ch = ["adv_email", "adv_online", "adv_community_presentation", "adv_other"]
    passive_ch = ["adv_poster", "adv_pamphlet", "adv_newsletter",
                  "adv_newspaper", "adv_radio", "adv_tv"]
    n_active = int(sum(adv[c] for c in active_ch))
    n_passive = int(sum(adv[c] for c in passive_ch))
    active_flag = int(n_active > 0)
    passive_only = int(n_active == 0 and n_passive > 0)
    digital_flag = int(adv["adv_email"] or adv["adv_online"])
    total_strat = n_active + n_passive
    ad_cost = int(sum(AD_CHANNELS[c]["cost"] for c in AD_CHANNELS if adv[c]))

    sf_base = 0.25 + 0.03 * p["protocol_complexity_score"]
    if p["biomarker_required"]:
        sf_base += 0.08
    if ec_type == "LEC":
        sf_base += 0.02
    sf_rate = float(np.clip(np.random.normal(sf_base, 0.05), 0.05, 0.75))

    ad_eff = sum(AD_CHANNELS[c]["effectiveness"] * adv[c] for c in AD_CHANNELS if adv[c])
    ad_mult = (1 + 0.35 * np.log1p(ad_eff)) if ad_eff > 0 else 0.85

    signal = (
        eligible * (1 - sf_rate) * (1 + 0.04 * pi_exp) * (1 + 0.06 * pi_prior)
        * (0.85 ** competing) * (0.90 ** (p["protocol_complexity_score"] - 5))
        * ad_mult * (window / 365) ** 0.6 * np.random.normal(1.0, 0.15)
    )
    enrolled = int(np.clip(signal, 0, 400))
    rate = round(enrolled / max(window / 30, 1), 2)

    ch_eff = {c: AD_CHANNELS[c]["effectiveness"] * adv[c] for c in AD_CHANNELS}
    bias = {
        "PROTO-A_ONCO":  {"adv_community_presentation": 1.3, "adv_email": 1.2, "adv_online": 0.9},
        "PROTO-B_CARDIO": {"adv_online": 1.3, "adv_email": 1.1, "adv_community_presentation": 1.0},
        "PROTO-C_NEURO": {"adv_community_presentation": 1.4, "adv_newsletter": 1.3, "adv_email": 1.0},
    }[pid]
    weighted = {c: ch_eff[c] * bias.get(c, 1.0) for c in ch_eff}
    best = max(weighted, key=weighted.get) if any(weighted.values()) else "none"

    return {
        "trial_id": f"SITE-{i:05d}", "protocol_id": pid, "country": country, "region": region,
        "therapeutic_area": p["therapeutic_area"], "phase": p["phase"],
        "indication": p["indication"], "sponsor_type": p["sponsor_type"],
        "protocol_complexity_score": p["protocol_complexity_score"],
        "num_visits": p["num_visits"], "num_inclusion_criteria": p["num_inclusion_criteria"],
        "num_exclusion_criteria": p["num_exclusion_criteria"],
        "biomarker_required": int(p["biomarker_required"]),
        "prior_therapy_restriction": int(p["prior_therapy_restriction"]),
        "placebo_controlled": int(p["placebo_controlled"]),
        "site_pi_id": f"PI-{np.random.randint(1, 30):03d}",
        "pi_experience_years": pi_exp, "pi_prior_trials_same_indication": pi_prior,
        "coordinator_count": coord, "gcp_training_valid": gcp,
        "competing_trials_same_indication_region": competing,
        "eligible_patient_pool_estimate": eligible,
        "expected_duration_years": dur,
        "estimated_start_date": est_start,
        "actual_first_patient_in": actual_fpi,
        "actual_last_patient_in": actual_lpi,
        "estimated_completion_date": est_completion,
        "fpi_delay_days": fpi_delay, "recruitment_window_days": window,
        "schedule_slip_days": slip, "start_quarter": quarter, "start_year": year,
        "regulatory_authority_type": authority,
        "regulatory_submission_date": reg_sub, "regulatory_approval_date": reg_app,
        "regulatory_delay_days": reg_app_delay, "import_license_required": int(import_license),
        "ec_type": ec_type, "ec_submission_date": ec_sub, "ec_approval_date": ec_app,
        "ec_approval_delay_days": ec_app_delay,
        "ec_submission_to_fpi_days": int((actual_fpi - ec_sub).days),
        "num_ec_amendments": amendments,
        **adv, "adv_other_specify": adv_other_specify,
        "num_active_strategies": n_active, "num_passive_strategies": n_passive,
        "total_strategies": total_strat, "active_outreach_flag": active_flag,
        "passive_only_flag": passive_only, "digital_flag": digital_flag,
        "ad_cost_total": ad_cost,
        "screen_failure_rate": round(sf_rate, 3),
        "actual_enrolled": enrolled, "enrollment_rate_per_month": rate,
        "best_channel": best, "best_channel_label": CHANNEL_LABEL.get(best, "None"),
    }

df = pd.DataFrame([generate_row(i) for i in range(N_ROWS)])
df.to_csv("site_trial_history_extended.csv", index=False)

ad_df = pd.DataFrame(AD_CHANNELS).T.reset_index().rename(columns={"index": "channel"})
ad_df["channel_label"] = ad_df["channel"].map(CHANNEL_LABEL)
ad_df.to_csv("advertisement_strategy_catalog.csv", index=False)

proto_df = pd.DataFrame(PROTOCOLS).T.reset_index().rename(columns={"index": "protocol_id"})
proto_df.to_csv("protocol_catalog.csv", index=False)

print(f"   ✅ {len(df)} rows × {df.shape[1]} cols saved")
print(f"   ✅ advertisement_strategy_catalog.csv")
print(f"   ✅ protocol_catalog.csv")

# ======================================================================
# STAGE 3 — FEATURE LISTS + MODELS
# ======================================================================
print("\n[Stage 3] Training models ...")

cat_cols = ["protocol_id", "country", "region", "therapeutic_area",
            "phase", "sponsor_type", "regulatory_authority_type", "ec_type"]

num_cols = [
    "protocol_complexity_score", "num_visits", "num_inclusion_criteria",
    "num_exclusion_criteria", "biomarker_required", "prior_therapy_restriction",
    "placebo_controlled", "pi_experience_years", "pi_prior_trials_same_indication",
    "coordinator_count", "gcp_training_valid",
    "competing_trials_same_indication_region", "eligible_patient_pool_estimate",
    "expected_duration_years", "fpi_delay_days", "recruitment_window_days",
    "schedule_slip_days", "start_quarter", "regulatory_delay_days",
    "import_license_required", "ec_approval_delay_days",
    "ec_submission_to_fpi_days", "num_ec_amendments",
    "adv_poster", "adv_pamphlet", "adv_email", "adv_online",
    "adv_newsletter", "adv_newspaper", "adv_radio", "adv_tv",
    "adv_community_presentation", "adv_other",
    "num_active_strategies", "num_passive_strategies", "total_strategies",
    "active_outreach_flag", "passive_only_flag", "digital_flag",
    "ad_cost_total", "screen_failure_rate",
]

df["protocol_label"] = df["protocol_id"].map(PROTO_LABEL)

X = df[cat_cols + num_cols]
y = df["actual_enrolled"]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols),
])
reg = Pipeline([
    ("prep", pre),
    ("gbr", GradientBoostingRegressor(n_estimators=500, learning_rate=0.05,
                                      max_depth=3, random_state=42)),
])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
reg.fit(Xtr, ytr)
pred = reg.predict(Xte)
mae = mean_absolute_error(yte, pred)
r2 = r2_score(yte, pred)
print(f"   Enrollment regressor — MAE = {mae:.2f}, R² = {r2:.3f}")
joblib.dump(reg, "site_enrollment_model.pkl")

df_cls = df[df["best_channel"] != "none"].copy()
Xc = df_cls[cat_cols + num_cols]
yc = df_cls["best_channel"]

clf = Pipeline([
    ("prep", pre),
    ("rf", RandomForestClassifier(n_estimators=500, max_depth=12,
                                  random_state=42, n_jobs=-1)),
])
Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(
    Xc, yc, test_size=0.2, random_state=42, stratify=yc
)
clf.fit(Xtr_c, ytr_c)
clf_acc = accuracy_score(yte_c, clf.predict(Xte_c))
print(f"   Best-channel classifier — Accuracy = {clf_acc:.3f}")
joblib.dump(clf, "best_channel_model.pkl")

# ======================================================================
# STAGE 4 — SLIDE VISUALS
# ======================================================================
print("\n[Stage 4] Generating slide visuals ...")
df["actual_first_patient_in"] = pd.to_datetime(df["actual_first_patient_in"])

# ---- SLIDE 1 ----
overview = pd.DataFrame({
    "Dimension": ["Total trial records", "Protocols covered", "Countries",
                  "Regulatory authorities", "Ethics committee types",
                  "Advertisement channels tracked", "Timeline features",
                  "Regulatory features", "Derived features", "Target variables"],
    "Value": [f"{len(df):,}", df['protocol_id'].nunique(), df['country'].nunique(),
              df['regulatory_authority_type'].nunique(), df['ec_type'].nunique(),
              len(CHANNELS), 10, 11, 21, "2 (enrollment, best channel)"],
})
overview.to_csv("slide1_overview.csv", index=False)
fig, ax = plt.subplots(figsize=(10, 5)); ax.axis("off")
tbl = ax.table(cellText=overview.values, colLabels=overview.columns,
               cellLoc="left", colLoc="left", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1, 1.6)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor(BRAND[0]); cell.set_text_props(color="white", weight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#F2F7FB")
plt.title("Model Dataset at a Glance", pad=20, weight="bold")
plt.savefig("slide1_overview.png"); plt.close()
print("   ✅ slide1_overview")

# ---- SLIDE 2 ----
fig, ax = plt.subplots(figsize=(10, 6))
order = list(PROTO_LABEL.values())
sns.boxplot(data=df, x="protocol_label", y="actual_enrolled",
            order=order, palette=BRAND[:3], ax=ax, width=0.55)
sns.stripplot(data=df, x="protocol_label", y="actual_enrolled",
              order=order, color="black", alpha=0.15, size=2, ax=ax)
ax.set_xlabel(""); ax.set_ylabel("Subjects enrolled")
ax.set_title("Enrollment distribution by protocol")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
plt.xticks(rotation=10)
plt.savefig("slide2_enrollment_box.png"); plt.close()
df.groupby("protocol_label")["actual_enrolled"].agg(
    ["count","mean","median","std","min","max"]).round(1).reset_index().to_csv(
    "slide2_enrollment_summary.csv", index=False)
print("   ✅ slide2_enrollment_box")

# ---- SLIDE 3 ----
fig, ax = plt.subplots(figsize=(10, 6))
for i, (label, grp) in enumerate(df.groupby("protocol_label")):
    ax.scatter(grp["recruitment_window_days"], grp["actual_enrolled"],
               alpha=0.45, s=22, color=BRAND[i], label=label)
    z = np.polyfit(grp["recruitment_window_days"], grp["actual_enrolled"], 1)
    xs = np.linspace(grp["recruitment_window_days"].min(),
                     grp["recruitment_window_days"].max(), 100)
    ax.plot(xs, np.polyval(z, xs), color=BRAND[i], lw=2)
ax.set_xlabel("Recruitment window (days, FPI → LPI)")
ax.set_ylabel("Subjects enrolled")
ax.set_title("Longer recruitment windows yield more subjects")
ax.legend(title="Protocol", frameon=True)
plt.savefig("slide3_window_vs_enrollment.png"); plt.close()
print("   ✅ slide3_window_vs_enrollment")

# ---- SLIDE 4 ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.boxplot(data=df, x="ec_type", y="fpi_delay_days",
            palette=BRAND[:3], ax=axes[0], width=0.5)
axes[0].set_title("FPI delay by ethics committee type")
axes[0].set_xlabel(""); axes[0].set_ylabel("Days from est. start to actual FPI")
sns.boxplot(data=df, x="ec_type", y="ec_approval_delay_days",
            palette=BRAND[:3], ax=axes[1], width=0.5)
axes[1].set_title("Ethics approval delay by committee type")
axes[1].set_xlabel(""); axes[1].set_ylabel("Days from submission to approval")
plt.tight_layout()
plt.savefig("slide4_ec_impact.png"); plt.close()
df.groupby("ec_type").agg(
    n=("trial_id","count"),
    mean_fpi_delay=("fpi_delay_days","mean"),
    median_fpi_delay=("fpi_delay_days","median"),
    mean_approval_delay=("ec_approval_delay_days","mean"),
    mean_amendments=("num_ec_amendments","mean"),
).round(1).reset_index().to_csv("slide4_ec_summary.csv", index=False)
print("   ✅ slide4_ec_impact")

# ---- SLIDE 5 ----
usage = df.groupby("protocol_label")[CHANNELS].mean().round(2)
usage.columns = [CHANNEL_LABEL[c] for c in usage.columns]
fig, ax = plt.subplots(figsize=(12, 4.5))
sns.heatmap(usage, annot=True, fmt=".0%", cmap="Blues",
            cbar_kws={"label": "% of trials using channel"}, ax=ax)
ax.set_title("Channel usage rate by protocol")
ax.set_xlabel(""); ax.set_ylabel("")
plt.savefig("slide5_channel_usage.png"); plt.close()
usage.to_csv("slide5_channel_usage.csv")
print("   ✅ slide5_channel_usage")

# ---- SLIDE 6 ----
effect = []
for label, grp in df.groupby("protocol_label"):
    for ch in CHANNELS:
        used = grp[grp[ch] == 1]["actual_enrolled"].mean()
        not_used = grp[grp[ch] == 0]["actual_enrolled"].mean()
        lift = (used - not_used) / not_used * 100 if not_used > 0 else 0
        effect.append({"Protocol": label, "Channel": CHANNEL_LABEL[ch], "Lift": lift})
pivot = pd.DataFrame(effect).pivot(index="Channel", columns="Protocol", values="Lift")
fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="RdYlGn", center=0,
            cbar_kws={"label": "% lift in enrollment"}, ax=ax)
ax.set_title("Enrollment lift by channel and protocol")
ax.set_xlabel(""); ax.set_ylabel("")
plt.savefig("slide6_channel_effectiveness.png"); plt.close()
pivot.round(1).to_csv("slide6_channel_effectiveness.csv")
print("   ✅ slide6_channel_effectiveness")

# ---- SLIDE 7 ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
df["outreach_type"] = np.where(df["active_outreach_flag"] == 1,
                               "Active outreach", "Passive only")
sns.boxplot(data=df, x="outreach_type", y="actual_enrolled",
            palette=[BRAND[4], BRAND[0]], ax=axes[0], width=0.5)
axes[0].set_title("Enrollment by outreach type")
axes[0].set_xlabel(""); axes[0].set_ylabel("Subjects enrolled")
strat = df.groupby("num_active_strategies")["actual_enrolled"].mean().reset_index()
sns.barplot(data=strat, x="num_active_strategies", y="actual_enrolled",
            palette="Blues_d", ax=axes[1])
axes[1].set_title("Diminishing returns with more active channels")
axes[1].set_xlabel("Number of active channels used")
axes[1].set_ylabel("Mean subjects enrolled")
for i, v in enumerate(strat["actual_enrolled"]):
    axes[1].text(i, v + 2, f"{v:.0f}", ha="center", fontsize=11, weight="bold")
plt.tight_layout()
plt.savefig("slide7_active_vs_passive.png"); plt.close()
df.groupby("outreach_type")["actual_enrolled"].agg(
    ["count","mean","median","std"]).round(1).reset_index().to_csv(
    "slide7_active_summary.csv", index=False)
print("   ✅ slide7_active_vs_passive")

# ---- SLIDE 8 ----
ohe = reg.named_steps["prep"].named_transformers_["cat"]
cat_names = ohe.get_feature_names_out(cat_cols)
all_names = list(cat_names) + num_cols
importances = reg.named_steps["gbr"].feature_importances_
imp_df = (pd.DataFrame({"Feature": all_names, "Importance": importances})
            .sort_values("Importance", ascending=False).head(15))
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=imp_df, y="Feature", x="Importance", palette="Blues_r", ax=ax)
ax.set_title("Top 15 drivers of site enrollment")
ax.set_xlabel("Relative importance"); ax.set_ylabel("")
plt.savefig("slide8_feature_importance.png"); plt.close()
imp_df.to_csv("slide8_feature_importance.csv", index=False)
print("   ✅ slide8_feature_importance")

# ---- SLIDE 9 ----
fig, ax = plt.subplots(figsize=(8.5, 8))
ax.scatter(yte, pred, alpha=0.4, s=25, color=BRAND[0], edgecolor="white")
lim = [0, max(yte.max(), pred.max()) * 1.05]
ax.plot(lim, lim, "--", color=BRAND[4], lw=2, label="Perfect prediction")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("Actual subjects enrolled")
ax.set_ylabel("Predicted subjects enrolled")
ax.set_title(f"Predicted vs. actual (R² = {r2:.2f}, MAE = {mae:.0f})")
ax.legend()
plt.savefig("slide9_pred_vs_actual.png"); plt.close()
pd.DataFrame({
    "Metric": ["R²","MAE (subjects)","Training rows","Test rows"],
    "Value": [round(r2,3), round(mae,1), len(Xtr), len(Xte)],
}).to_csv("slide9_model_accuracy.csv", index=False)
print("   ✅ slide9_pred_vs_actual")

# ---- SLIDE 10 — with Option A fix ----
baseline_rows = (
    df.groupby("protocol_id")[num_cols]
      .median(numeric_only=True)
      .reset_index()
)
for c in cat_cols:
    mode_val = df.groupby("protocol_id")[c].agg(lambda s: s.mode().iloc[0])
    baseline_rows[c] = baseline_rows["protocol_id"].map(mode_val)

new_site_overrides = {
    "country": "Germany", "region": "Europe", "ec_type": "CEC",
    "regulatory_authority_type": "EMA",
    "eligible_patient_pool_estimate": 350,
    "competing_trials_same_indication_region": 3,
    "pi_experience_years": 12, "pi_prior_trials_same_indication": 4,
    "coordinator_count": 3, "fpi_delay_days": 90,
    "recruitment_window_days": 540, "ec_approval_delay_days": 60,
    "num_ec_amendments": 1, "screen_failure_rate": 0.35,
}
new_df = baseline_rows.copy()
for k, v in new_site_overrides.items():
    new_df[k] = v

missing = [c for c in (cat_cols + num_cols) if c not in new_df.columns]
assert not missing, f"Missing columns: {missing}"

proba = clf.predict_proba(new_df[cat_cols + num_cols])
classes = clf.classes_
top3 = np.argsort(-proba, axis=1)[:, :3]

rec_rows = []
for i, pid in enumerate(new_df["protocol_id"]):
    rec_rows.append({
        "Protocol": PROTO_LABEL[pid],
        "Top channel": CHANNEL_LABEL.get(classes[top3[i, 0]], classes[top3[i, 0]]),
        "Confidence": f"{proba[i, top3[i, 0]]:.0%}",
        "2nd choice": CHANNEL_LABEL.get(classes[top3[i, 1]], classes[top3[i, 1]]),
        "3rd choice": CHANNEL_LABEL.get(classes[top3[i, 2]], classes[top3[i, 2]]),
    })
rec_df = pd.DataFrame(rec_rows)
rec_df.to_csv("slide10_recommendations.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 3.5)); ax.axis("off")
tbl = ax.table(cellText=rec_df.values, colLabels=rec_df.columns,
               cellLoc="left", colLoc="left", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1, 1.9)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor(BRAND[0]); cell.set_text_props(color="white", weight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#F2F7FB")
plt.title(f"Recommended recruitment strategy per protocol "
          f"(model accuracy = {clf_acc:.0%})", pad=18, weight="bold")
plt.savefig("slide10_recommendations.png"); plt.close()
print("   ✅ slide10_recommendations")

# ---- SLIDE 11 ----
catalog = pd.read_csv("advertisement_strategy_catalog.csv", index_col="channel")
rows = []
for ch in CHANNELS:
    used_mask = df[ch] == 1
    n_used = int(used_mask.sum())
    if n_used == 0:
        continue
    mean_used = df.loc[used_mask, "actual_enrolled"].mean()
    mean_not_used = df.loc[~used_mask, "actual_enrolled"].mean()
    lift = mean_used - mean_not_used
    cost = float(catalog.loc[ch, "cost"])
    rows.append({
        "Channel": CHANNEL_LABEL[ch], "Channel_key": ch, "N trials": n_used,
        "Mean enrolled (used)": round(mean_used, 1),
        "Mean enrolled (not used)": round(mean_not_used, 1),
        "Lift (subjects)": round(lift, 1), "Cost (USD)": cost,
        "Lift per $1,000": round(lift / cost * 1000, 2) if cost > 0 else np.nan,
    })
cost_df = pd.DataFrame(rows).sort_values("Lift per $1,000", ascending=False)

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=cost_df, x="Lift per $1,000", y="Channel",
            palette="Greens_r", ax=ax)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Cost-efficiency of recruitment channels\n"
             "(incremental subjects enrolled per $1,000 spent)")
ax.set_xlabel("Additional subjects per $1,000"); ax.set_ylabel("")
for i, (_, row) in enumerate(cost_df.iterrows()):
    ax.text(row["Lift per $1,000"], i,
            f"  {row['Lift per $1,000']:.2f}", va="center", fontsize=10)
plt.tight_layout()
plt.savefig("slide11_cost_efficiency.png"); plt.close()
cost_df.to_csv("slide11_cost_efficiency.csv", index=False)
print("   ✅ slide11_cost_efficiency")

# ---- SLIDE 12 ----
country_tbl = df.groupby("country").agg(
    n_trials=("trial_id", "count"),
    mean_enrolled=("actual_enrolled", "mean"),
    median_enrolled=("actual_enrolled", "median"),
    mean_fpi_delay=("fpi_delay_days", "mean"),
    mean_window=("recruitment_window_days", "mean"),
).round(1).sort_values("mean_enrolled", ascending=False).reset_index()
country_tbl.to_csv("slide12_country_summary.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=country_tbl, y="country", x="mean_enrolled",
            palette="Blues_r", ax=axes[0])
axes[0].set_title("Mean enrollment by country")
axes[0].set_xlabel("Mean subjects enrolled"); axes[0].set_ylabel("")
sns.scatterplot(data=country_tbl, x="mean_fpi_delay", y="mean_enrolled",
                size="n_trials", sizes=(80, 600), color=BRAND[0],
                ax=axes[1], legend="brief")
for _, row in country_tbl.iterrows():
    axes[1].annotate(row["country"], (row["mean_fpi_delay"], row["mean_enrolled"]),
                     xytext=(5, 5), textcoords="offset points", fontsize=10)
axes[1].set_title("Speed vs. yield: country trade-off")
axes[1].set_xlabel("Mean FPI delay (days)")
axes[1].set_ylabel("Mean subjects enrolled")
plt.tight_layout()
plt.savefig("slide12_country_performance.png"); plt.close()
print("   ✅ slide12_country_performance")

# ======================================================================
# DONE
# ======================================================================
print("\n" + "=" * 70)
print("🎉 PIPELINE COMPLETE")
print("=" * 70)
print(f"Working directory   : {os.getcwd()}")
print(f"Master dataset      : site_trial_history_extended.csv ({len(df):,} rows)")
print(f"Trained models      : site_enrollment_model.pkl, best_channel_model.pkl")
print(f"Regressor R²        : {r2:.3f}  (MAE = {mae:.1f} subjects)")
print(f"Classifier accuracy : {clf_acc:.0%}")
print(f"Slide visuals       : 12 PNG + 12 CSV")
print("=" * 70)
print("\nTop-3 recommended channels per protocol:")
print(rec_df.to_string(index=False))
print("=" * 70)

Mounted at /content/drive
📁 Working directory: /content/drive/MyDrive/HealthCare_/CTR_Recruitment/DataSource
CLINICAL TRIAL RECRUITMENT — FULL PIPELINE

[Stage 2] Generating synthetic dataset ...
   ✅ 3000 rows × 66 cols saved
   ✅ advertisement_strategy_catalog.csv
   ✅ protocol_catalog.csv

[Stage 3] Training models ...
   Enrollment regressor — MAE = 12.52, R² = 0.983
   Best-channel classifier — Accuracy = 0.995

[Stage 4] Generating slide visuals ...
   ✅ slide1_overview


/tmp/ipykernel_3256/2935890990.py:405: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="protocol_label", y="actual_enrolled",


   ✅ slide2_enrollment_box
   ✅ slide3_window_vs_enrollment


/tmp/ipykernel_3256/2935890990.py:437: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="ec_type", y="fpi_delay_days",
/tmp/ipykernel_3256/2935890990.py:441: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="ec_type", y="ec_approval_delay_days",


   ✅ slide4_ec_impact
   ✅ slide5_channel_usage
   ✅ slide6_channel_effectiveness


...

   ✅ slide8_feature_importance
   ✅ slide9_pred_vs_actual
   ✅ slide10_recommendations


/tmp/ipykernel_3256/2935890990.py:620: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=cost_df, x="Lift per $1,000", y="Channel",


   ✅ slide11_cost_efficiency


/tmp/ipykernel_3256/2935890990.py:645: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=country_tbl, y="country", x="mean_enrolled",


   ✅ slide12_country_performance

🎉 PIPELINE COMPLETE
Working directory   : /content/drive/MyDrive/HealthCare_/CTR_Recruitment/DataSource
Master dataset      : site_trial_history_extended.csv (3,000 rows)
Trained models      : site_enrollment_model.pkl, best_channel_model.pkl
Regressor R²        : 0.983  (MAE = 12.5 subjects)
Classifier accuracy : 99%
Slide visuals       : 12 PNG + 12 CSV

Top-3 recommended channels per protocol:
                    Protocol Top channel Confidence 2nd choice 3rd choice
A · Oncology (NSCLC, Ph III)      Online        41%      Email  Community
  B · Cardiology (HF, Ph II)       Email        92%     Online  Community
   C · Neurology (AD, Ph II)      Online        45%      Email  Community
